In [1]:
import gymnasium as gym
import numpy as np
import random
from collections import defaultdict

# -----------------------------
# a) Create Deterministic FrozenLake
# -----------------------------
env = gym.make("FrozenLake-v1", is_slippery=False)

n_states = env.observation_space.n
n_actions = env.action_space.n

# -----------------------------
# Q(s,a) and Returns Storage
# -----------------------------
Q = np.zeros((n_states, n_actions))
returns = defaultdict(list)   # Stores returns for First Visit MC

# -----------------------------
# Hyperparameters
# -----------------------------
episodes = 20000
gamma = 0.95

epsilon = 1.0
epsilon_decay = 0.9995
epsilon_min = 0.05

# -----------------------------
# epsilon-Greedy Policy
# -----------------------------
def epsilon_greedy(state):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(Q[state])

# -----------------------------
# b) First-Visit Monte Carlo
# -----------------------------
print("Training using First-Visit Monte Carlo...\n")

for ep in range(episodes):

    episode = []  # store (state, action, reward)

    state, _ = env.reset()

    done = False

    # ---- Generate Episode ----
    while not done:
        action = epsilon_greedy(state)

        next_state, reward, terminated, truncated, _ = env.step(action)

        episode.append((state, action, reward))

        state = next_state
        done = terminated or truncated

    # ---- Compute Returns (Backward Pass) ----
    G = 0
    visited = set()

    for t in reversed(range(len(episode))):
        state_t, action_t, reward_t = episode[t]
        G = gamma * G + reward_t

        # First-Visit check
        if (state_t, action_t) not in visited:
            visited.add((state_t, action_t))

            returns[(state_t, action_t)].append(G)
            Q[state_t, action_t] = np.mean(returns[(state_t, action_t)])

    # ---- Decay Exploration ----
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep + 1) % 5000 == 0:
        print(f"Episode {ep+1}/{episodes}, Epsilon={epsilon:.4f}")

print("\nTraining Complete")

# -----------------------------
# c) Learned Optimal Policy
# -----------------------------
policy = np.argmax(Q, axis=1)

# -----------------------------
# d) Display Policy
# -----------------------------
action_map = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

print("\nLearned Policy (4x4 Grid):\n")

for i in range(4):
    row = ""
    for j in range(4):
        s = i * 4 + j
        row += action_map[policy[s]] + " "
    print(row)

env.close()

Training using First-Visit Monte Carlo...

Episode 5000/20000, Epsilon=0.0820
Episode 10000/20000, Epsilon=0.0500
Episode 15000/20000, Epsilon=0.0500
Episode 20000/20000, Epsilon=0.0500

Training Complete

Learned Policy (4x4 Grid):

↓ → ↓ ← 
↓ ← ↓ ← 
→ ↓ ↓ ← 
← → → ← 


In [3]:
#visalize using gymnasium
import time
env = gym.make("FrozenLake-v1", is_slippery=False)
print("\nStarting Visualization...")
state, _ = env.reset()
done = False
while not done:
    action = policy[state]
    env.render()
    state, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.5)  # Slow down so you can see movement
    done = terminated or truncated
env.close()


Starting Visualization...
